# OpenMC depletion benchmark — BEAVRS 2.4% PWR pincell (Aegis-40 Digital Appendix)

**What this is.** A *confirmatory reproduction* of the published OpenMC depletion benchmark
(Romano et al., *Ann. Nucl. Energy* **152** (2021) 107989, §3.2 — PWR pincell), run with **our**
ENDF/B-VIII.0 library and depletion chain.

**What it proves (be honest in the FER):** it does **not** independently validate OpenMC's physics
— Romano 2021 already did that against Serpent (k_eff within ~20 pcm, actinides/FPs <1%), and *that*
we cite. Running the built-in pincell ourselves proves the narrower, necessary thing the competition's
Digital-Appendix gate actually asks for:
1. **Toolchain-configuration acceptance** — our install + our cross-section build + our chain file +
   our units/volume/power reproduce the published k(BU) curve and isotopic trends. Catches the real
   failure modes (wrong library, wrong chain, broken coupling, mis-set burnup units).
2. A **sample input file we actually ran** (reproducibility) + a **seed-repeat leg** (repeatability).
3. The basis for the code/data **bias term Δ_bias** fed into `run_storage_criticality.py`.

**Scope here = Option C (trimmed).** Enough statistics for a clean k(BU) curve at **σ_k ≈ 25–30 pcm**
(plenty for a configuration check — a real misconfiguration is hundreds-to-thousands of pcm; we do **not**
chase the cited 20 pcm code-to-code bias), and a **coarser burnup schedule to ~31 MWd/kg** instead of the
full 50 — the late high-burnup steps are the slow ones and add little here. Tune the knobs in cell **[2]**.

**I/O fix already applied:** the 13 GB library + chain were copied from `/mnt/d` (slow WSL 9p mount)
to native ext4 `~/openmc_data/`. The cells below point there. This is also what makes your real
STAT_FINAL core run faster.

---
**Before running:** launch Jupyter from the OpenMC conda env so this notebook's kernel can import
OpenMC:
```bash
source ~/miniconda3/etc/profile.d/conda.sh
conda activate /mnt/d/conda-envs/openmc-py311
cd /mnt/d/projects/teknofest-2026-aegis-40-ipwr
jupyter lab scripts/beavrs_pincell_benchmark.ipynb
```

## 1. Environment + data paths (ext4)

In [1]:
import os, sys, time, math
import numpy as np
import openmc
import openmc.deplete

# --- ext4 copies (fast). Fall back to /mnt/d only if the ext4 copy is absent. ---
EXT4 = os.path.expanduser('~/openmc_data')
XS_EXT4    = os.path.join(EXT4, 'endfb-viii.0-hdf5', 'cross_sections.xml')
CHAIN_EXT4 = os.path.join(EXT4, 'chain_endfb80_pwr.xml')
XS_MNT     = '/mnt/d/openmc_data/endfb-viii.0-hdf5/cross_sections.xml'
CHAIN_MNT  = '/mnt/d/openmc_data/chain_endfb80_pwr.xml'

XS    = XS_EXT4    if os.path.exists(XS_EXT4)    else XS_MNT
CHAIN = CHAIN_EXT4 if os.path.exists(CHAIN_EXT4) else CHAIN_MNT
assert os.path.exists(XS),    f'cross_sections.xml not found: {XS}'
assert os.path.exists(CHAIN), f'chain not found: {CHAIN}'

openmc.config['cross_sections'] = XS
openmc.config['chain_file']     = CHAIN

print('python :', sys.executable)
print('openmc :', openmc.__version__)
print('xs     :', XS, '  (ext4)' if XS == XS_EXT4 else '  (SLOW /mnt/d — copy not finished?)')
print('chain  :', CHAIN, '(ext4)' if CHAIN == CHAIN_EXT4 else '(SLOW /mnt/d)')

python : /mnt/d/conda-envs/openmc-py311/bin/python
openmc : 0.15.3
xs     : /home/samira/openmc_data/endfb-viii.0-hdf5/cross_sections.xml   (ext4)
chain  : /home/samira/openmc_data/chain_endfb80_pwr.xml (ext4)


## 2. Knobs — statistics & burnup schedule

- **Statistics** buys *k-resolution*. We are running a **configuration-acceptance reproduction**, not
  re-deriving the ~20 pcm OpenMC-vs-Serpent bias (that we cite from Romano 2021). A real misconfiguration
  shows up as **hundreds-to-thousands of pcm**, never a subtle 20 pcm shift — so the working target is
  **σ_k ≈ 25–30 pcm** (rigorous-looking ≈ 20). You do **not** need 8 pcm / 200k particles.
  `40000 × (150−30) = 4.8 M` histories → σ_k ≈ 15–18 pcm, laptop-friendly.
- **Calibrate from the first run:** read the printed BOL σ, then scale `N_new = N × (σ / σ_target)²`.
- **Isotopics** (<1% claim) converge much faster than k — they do **not** need the high count.
- **Schedule** (Option C): fine start to capture Xe, then to ~31 MWd/kg. Set `FULL_50 = True` for the
  complete 50 MWd/kg / 28-step paper schedule (much slower).

> **Laptop tip:** do a first pass at `PARTICLES = 40_000` to time one full depletion and read the BOL σ.
> If σ is already ≤ ~25 pcm (likely), you're done — no need to go higher.

In [2]:
PARTICLES = 40000       # per batch — laptop-friendly (~σ 15–18 pcm). Raise only if you want tighter.
BATCHES   = 150
INACTIVE  = 30           # active = BATCHES - INACTIVE = 120
SEED      = 1            # change to e.g. 7 for the repeatability leg, then re-run cells 3–5

POWER_W_PER_CM = 174.0   # Romano 2021 §3.2 linear heat rate
FUEL_RADIUS_CM = 0.39218 # BEAVRS pellet radius (pwr_pin_cell)
FULL_50 = False          # False = Option C (~31 MWd/kg); True = full 50 MWd/kg paper schedule

if FULL_50:
    STEPS = [0.1, 0.4, 0.5] + [1.0]*9 + [2.5]*16          # → 50 MWd/kg, 28 steps
else:
    STEPS = [0.1, 0.4, 0.5] + [1.0]*5 + [5.0]*5           # → 31 MWd/kg, 13 steps (Option C)

BU_AXIS = np.concatenate([[0.0], np.cumsum(STEPS)])
active = BATCHES - INACTIVE
print(f'{len(STEPS)} steps → {BU_AXIS[-1]:.1f} MWd/kg | {PARTICLES:,} part × {active} active '
      f'= {PARTICLES*active/1e6:.1f} M histories/step | seed {SEED}')

13 steps → 31.0 MWd/kg | 40,000 part × 120 active = 4.8 M histories/step | seed 1


## 3. Build the BEAVRS pincell (robust fuel detection)

`pwr_pin_cell()` names the fuel `'UO2 (2.4%)'` (no literal "fuel"), so identify it by composition
(uranium + oxygen) — the bug that silently broke the first run.

In [3]:
def is_fuel(mat):
    name = (mat.name or '').lower()
    if 'uo2' in name or 'fuel' in name:
        return True
    nucs = {n for n, _, _ in mat.nuclides}
    return any(n.startswith('U23') for n in nucs) and any(n.startswith('O1') for n in nucs)

model = openmc.examples.pwr_pin_cell()
for mat in model.materials:
    tag = 'FUEL' if is_fuel(mat) else ''
    if is_fuel(mat):
        mat.volume = math.pi * FUEL_RADIUS_CM**2   # per 1 cm of pin height
        mat.depletable = True
    print(f'  id {mat.id}: {mat.name!r:24} {tag}')

model.settings.particles = PARTICLES
model.settings.batches   = BATCHES
model.settings.inactive  = INACTIVE
model.settings.seed      = SEED
fuel_id = next(m.id for m in model.materials if is_fuel(m))
print('fuel material id =', fuel_id)

  id 1: 'UO2 (2.4%)'             FUEL
  id 2: 'Zircaloy'               
  id 3: 'Hot borated water'      
fuel material id = 1


## 4. Run the depletion (live output)

OpenMC prints per-batch k_eff and per-step progress below as it runs. On ext4 each step should be
fast at first and slow down as fission products accumulate. `CoupledOperator` + `PredictorIntegrator`
= the paper's constant-extrapolation (CE) scheme.

In [ ]:
WORKDIR = os.path.join('docs', 'competition', 'digital-appendix', 'pincell_run')
os.makedirs(WORKDIR, exist_ok=True)

t0 = time.time()
cwd = os.getcwd()
os.chdir(WORKDIR)
try:
    op = openmc.deplete.CoupledOperator(model, chain_file=CHAIN)
    integrator = openmc.deplete.PredictorIntegrator(
        op, STEPS, power=POWER_W_PER_CM, timestep_units='MWd/kg')
    integrator.integrate()
finally:
    os.chdir(cwd)
print(f'\n[done] depletion wall time = {(time.time()-t0)/60:.1f} min')

                                %%%%%%%%%%%%%%%
                           %%%%%%%%%%%%%%%%%%%%%%%%
                        %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                      %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                    %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                   %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                                    %%%%%%%%%%%%%%%%%%%%%%%%
                                     %%%%%%%%%%%%%%%%%%%%%%%%
                 ###############      %%%%%%%%%%%%%%%%%%%%%%%%
                ##################     %%%%%%%%%%%%%%%%%%%%%%%
                ###################     %%%%%%%%%%%%%%%%%%%%%%%
                ####################     %%%%%%%%%%%%%%%%%%%%%%
                #####################     %%%%%%%%%%%%%%%%%%%%%
                ######################     %%%%%%%%%%%%%%%%%%%%
                #######################     %%%%%%%%%%%%%%%%%%
                 #######################     %%%%%%%%%%%%%%%%%
                 #####################

 Reading c_H_in_H2O from
 /home/samira/openmc_data/endfb-viii.0-hdf5/thermal/c_H_in_H2O.h5
 Minimum neutron data temperature: 294 K
 Maximum neutron data temperature: 294 K
 Preparing distributed cell instances...
 Reading plot XML file...
 Writing summary.h5 file...
[openmc.deplete] t=0.0 s, dt=217858.95191724383 s, source=174.0
 Reading H2 from /home/samira/openmc_data/endfb-viii.0-hdf5/neutron/H2.h5
 Reading H3 from /home/samira/openmc_data/endfb-viii.0-hdf5/neutron/H3.h5
 Reading He3 from /home/samira/openmc_data/endfb-viii.0-hdf5/neutron/He3.h5
 Reading He4 from /home/samira/openmc_data/endfb-viii.0-hdf5/neutron/He4.h5
 Reading Li6 from /home/samira/openmc_data/endfb-viii.0-hdf5/neutron/Li6.h5
 Reading Li7 from /home/samira/openmc_data/endfb-viii.0-hdf5/neutron/Li7.h5
 Reading Be7 from /home/samira/openmc_data/endfb-viii.0-hdf5/neutron/Be7.h5
 Reading Be9 from /home/samira/openmc_data/endfb-viii.0-hdf5/neutron/Be9.h5
 Reading C12 from /home/samira/openmc_data/endfb-viii.0-hdf5/neu

 Reading Na23 from /home/samira/openmc_data/endfb-viii.0-hdf5/neutron/Na23.h5
 Reading Mg24 from /home/samira/openmc_data/endfb-viii.0-hdf5/neutron/Mg24.h5
 Reading Mg25 from /home/samira/openmc_data/endfb-viii.0-hdf5/neutron/Mg25.h5
 Reading Mg26 from /home/samira/openmc_data/endfb-viii.0-hdf5/neutron/Mg26.h5
 Reading Al26_m1 from
 /home/samira/openmc_data/endfb-viii.0-hdf5/neutron/Al26_m1.h5
 Reading Al27 from /home/samira/openmc_data/endfb-viii.0-hdf5/neutron/Al27.h5
 Reading Si28 from /home/samira/openmc_data/endfb-viii.0-hdf5/neutron/Si28.h5
 Reading Si29 from /home/samira/openmc_data/endfb-viii.0-hdf5/neutron/Si29.h5
 Reading Si30 from /home/samira/openmc_data/endfb-viii.0-hdf5/neutron/Si30.h5
 Reading Si31 from /home/samira/openmc_data/endfb-viii.0-hdf5/neutron/Si31.h5
 Reading Si32 from /home/samira/openmc_data/endfb-viii.0-hdf5/neutron/Si32.h5
 Reading P31 from /home/samira/openmc_data/endfb-viii.0-hdf5/neutron/P31.h5
 Reading S32 from /home/samira/openmc_data/endfb-viii.0-hdf

 Reading Ar38 from /home/samira/openmc_data/endfb-viii.0-hdf5/neutron/Ar38.h5
 Reading Ar39 from /home/samira/openmc_data/endfb-viii.0-hdf5/neutron/Ar39.h5
 Reading Ar40 from /home/samira/openmc_data/endfb-viii.0-hdf5/neutron/Ar40.h5
 Reading Ar41 from /home/samira/openmc_data/endfb-viii.0-hdf5/neutron/Ar41.h5
 Reading K39 from /home/samira/openmc_data/endfb-viii.0-hdf5/neutron/K39.h5
 Reading K40 from /home/samira/openmc_data/endfb-viii.0-hdf5/neutron/K40.h5
 Reading K41 from /home/samira/openmc_data/endfb-viii.0-hdf5/neutron/K41.h5
 Reading Ca40 from /home/samira/openmc_data/endfb-viii.0-hdf5/neutron/Ca40.h5
 Reading Ca41 from /home/samira/openmc_data/endfb-viii.0-hdf5/neutron/Ca41.h5
 Reading Ca42 from /home/samira/openmc_data/endfb-viii.0-hdf5/neutron/Ca42.h5
 Reading Ca43 from /home/samira/openmc_data/endfb-viii.0-hdf5/neutron/Ca43.h5
 Reading Ca44 from /home/samira/openmc_data/endfb-viii.0-hdf5/neutron/Ca44.h5
 Reading Ca45 from /home/samira/openmc_data/endfb-viii.0-hdf5/neutron/

 Reading Se81 from /home/samira/openmc_data/endfb-viii.0-hdf5/neutron/Se81.h5
 Reading Se82 from /home/samira/openmc_data/endfb-viii.0-hdf5/neutron/Se82.h5
 Reading Br79 from /home/samira/openmc_data/endfb-viii.0-hdf5/neutron/Br79.h5
 Reading Br80 from /home/samira/openmc_data/endfb-viii.0-hdf5/neutron/Br80.h5
 Reading Br81 from /home/samira/openmc_data/endfb-viii.0-hdf5/neutron/Br81.h5
 Reading Kr78 from /home/samira/openmc_data/endfb-viii.0-hdf5/neutron/Kr78.h5
 Reading Kr79 from /home/samira/openmc_data/endfb-viii.0-hdf5/neutron/Kr79.h5
 Reading Kr80 from /home/samira/openmc_data/endfb-viii.0-hdf5/neutron/Kr80.h5
 Reading Kr81 from /home/samira/openmc_data/endfb-viii.0-hdf5/neutron/Kr81.h5
 Reading Kr82 from /home/samira/openmc_data/endfb-viii.0-hdf5/neutron/Kr82.h5
 Reading Kr83 from /home/samira/openmc_data/endfb-viii.0-hdf5/neutron/Kr83.h5
 Reading Kr84 from /home/samira/openmc_data/endfb-viii.0-hdf5/neutron/Kr84.h5
 Reading Kr85 from /home/samira/openmc_data/endfb-viii.0-hdf5/ne

...

          1200K
          2500K


 Reading Lu176 from /home/samira/openmc_data/endfb-viii.0-hdf5/neutron/Lu176.h5
 Reading Hf174 from /home/samira/openmc_data/endfb-viii.0-hdf5/neutron/Hf174.h5
 Reading Hf175 from /home/samira/openmc_data/endfb-viii.0-hdf5/neutron/Hf175.h5
 Reading Hf176 from /home/samira/openmc_data/endfb-viii.0-hdf5/neutron/Hf176.h5
 Reading Hf177 from /home/samira/openmc_data/endfb-viii.0-hdf5/neutron/Hf177.h5
 Reading Hf178 from /home/samira/openmc_data/endfb-viii.0-hdf5/neutron/Hf178.h5
 Reading Hf179 from /home/samira/openmc_data/endfb-viii.0-hdf5/neutron/Hf179.h5
 Reading Hf180 from /home/samira/openmc_data/endfb-viii.0-hdf5/neutron/Hf180.h5
 Reading Hf181 from /home/samira/openmc_data/endfb-viii.0-hdf5/neutron/Hf181.h5
 Reading Hf182 from /home/samira/openmc_data/endfb-viii.0-hdf5/neutron/Hf182.h5
 Reading Ta180 from /home/samira/openmc_data/endfb-viii.0-hdf5/neutron/Ta180.h5


          1200K
          2500K
          1200K
          2500K


 Reading Ta181 from /home/samira/openmc_data/endfb-viii.0-hdf5/neutron/Ta181.h5
 Reading Ta182 from /home/samira/openmc_data/endfb-viii.0-hdf5/neutron/Ta182.h5
 Reading W180 from /home/samira/openmc_data/endfb-viii.0-hdf5/neutron/W180.h5
 Reading W181 from /home/samira/openmc_data/endfb-viii.0-hdf5/neutron/W181.h5
 Reading W182 from /home/samira/openmc_data/endfb-viii.0-hdf5/neutron/W182.h5
 Reading W183 from /home/samira/openmc_data/endfb-viii.0-hdf5/neutron/W183.h5
 Reading W184 from /home/samira/openmc_data/endfb-viii.0-hdf5/neutron/W184.h5
 Reading W185 from /home/samira/openmc_data/endfb-viii.0-hdf5/neutron/W185.h5
 Reading W186 from /home/samira/openmc_data/endfb-viii.0-hdf5/neutron/W186.h5
 Reading Re185 from /home/samira/openmc_data/endfb-viii.0-hdf5/neutron/Re185.h5
 Reading Re186_m1 from
 /home/samira/openmc_data/endfb-viii.0-hdf5/neutron/Re186_m1.h5
 Reading Re187 from /home/samira/openmc_data/endfb-viii.0-hdf5/neutron/Re187.h5
 Reading Os184 from /home/samira/openmc_data/en

 Reading Cf251 from /home/samira/openmc_data/endfb-viii.0-hdf5/neutron/Cf251.h5
 Reading Cf252 from /home/samira/openmc_data/endfb-viii.0-hdf5/neutron/Cf252.h5
 Reading Cf253 from /home/samira/openmc_data/endfb-viii.0-hdf5/neutron/Cf253.h5
 Reading Cf254 from /home/samira/openmc_data/endfb-viii.0-hdf5/neutron/Cf254.h5
 Reading Es251 from /home/samira/openmc_data/endfb-viii.0-hdf5/neutron/Es251.h5
 Reading Es252 from /home/samira/openmc_data/endfb-viii.0-hdf5/neutron/Es252.h5
 Reading Es253 from /home/samira/openmc_data/endfb-viii.0-hdf5/neutron/Es253.h5
 Reading Es254 from /home/samira/openmc_data/endfb-viii.0-hdf5/neutron/Es254.h5
 Reading Es254_m1 from
 /home/samira/openmc_data/endfb-viii.0-hdf5/neutron/Es254_m1.h5
 Reading Es255 from /home/samira/openmc_data/endfb-viii.0-hdf5/neutron/Es255.h5
 Reading Fm255 from /home/samira/openmc_data/endfb-viii.0-hdf5/neutron/Fm255.h5
 Maximum neutron transport energy: 20000000 eV for Zr90
 Initializing source particles...

 ====================>

## 5. Results — k_eff(BU) with statistical σ

In [ ]:
res = openmc.deplete.Results(os.path.join(WORKDIR, 'depletion_results.h5'))
_, keff = res.get_keff()           # shape (N, 2) = [value, std]
k   = keff[:, 0]
sig = keff[:, 1]

print(f'{"BU (MWd/kg)":>12} | {"k_eff":>9} | {"sigma (pcm)":>11}')
print('-'*40)
for b, kk, ss in zip(BU_AXIS, k, sig):
    print(f'{b:12.2f} | {kk:9.5f} | {ss*1e5:11.0f}')
print('-'*40)
sig0 = sig[0]*1e5
print(f'BOL k = {k[0]:.5f} ± {sig0:.0f} pcm   (target σ ≈ 25–30 pcm for the configuration check)')
print(f'EOL k = {k[-1]:.5f} at {BU_AXIS[-1]:.1f} MWd/kg')
SIGMA_TARGET = 25.0
if sig0 > SIGMA_TARGET:
    factor = (sig0 / SIGMA_TARGET)**2
    print(f'note: σ {sig0:.0f} pcm > {SIGMA_TARGET:.0f}. To reach target, set '
          f'PARTICLES ≈ {int(PARTICLES*factor):,} and re-run cells 3–5 (or accept — still fine '
          f'for a config check, a real error would be >>100 pcm).')
else:
    print(f'σ {sig0:.0f} pcm ≤ {SIGMA_TARGET:.0f} pcm — sufficient for the configuration-acceptance claim.')

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7, 4))
ax.errorbar(BU_AXIS, k, yerr=sig, fmt='o-', capsize=3, lw=1.2)
ax.set_xlabel('Burnup (MWd/kg)'); ax.set_ylabel('k_eff')
ax.set_title('BEAVRS 2.4% pincell — OpenMC depletion (ENDF/B-VIII.0)')
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(WORKDIR, 'pincell_keff_vs_burnup.png'), dpi=140)
plt.show()

## 6. Principal isotopics (the <1% claim)

U-235 depletion + Pu-239/240/241 in-growth + Cs-137/Sr-90 accumulation are the physical signatures
the paper benchmarks to <1%. These feed the storage-criticality k(95/95) and the source-term/decay-heat
path.

In [ ]:
nucs = ['U235', 'Pu239', 'Pu240', 'Pu241', 'Am241', 'Cs137', 'Sr90', 'Sm149']
conc = {}
for n in nucs:
    try:
        _, atoms = res.get_atoms(str(fuel_id), n)
        conc[n] = np.asarray(atoms)
    except Exception:
        conc[n] = None

fig, ax = plt.subplots(figsize=(7, 4))
for n in nucs:
    if conc[n] is not None and conc[n].max() > 0:
        ax.plot(BU_AXIS, conc[n] / conc[n].max(), 'o-', ms=3, label=n)
ax.set_xlabel('Burnup (MWd/kg)'); ax.set_ylabel('atoms (normalised to each nuclide max)')
ax.set_title('Principal-isotope build-up / depletion'); ax.legend(ncol=2, fontsize=8)
ax.grid(alpha=0.3); fig.tight_layout()
fig.savefig(os.path.join(WORKDIR, 'pincell_isotopics.png'), dpi=140)
plt.show()

## 7. Acceptance & how it feeds the FER

- **Benchmark target (Romano 2021, cited):** OpenMC vs Serpent — k_eff ~20 pcm, actinides <1%, FPs <1%.
  Reproducing the k(BU) curve + isotopic trends with our library/chain demonstrates the depletion path
  is **configured and used correctly**.
- **Reproducibility:** model is `openmc.examples.pwr_pin_cell()` — no proprietary geometry; anyone reruns
  it exactly.
- **Repeatability:** set `SEED = 7` in cell [Knobs], re-run cells 3–5; k at each step must agree within
  the combined Monte-Carlo σ.
- **Δ_bias:** the spread vs the reference is the depletion contribution to the code/data bias in
  `run_storage_criticality.py` (currently 0). At ~20 pcm / <1% it is small vs the ~0.16 storage margin.
- **§8.13 wording:** describe this as *“confirmatory reproduction of the published BEAVRS pincell
  benchmark with our ENDF/B-VIII.0 library and chain,”* **not** “validation of OpenMC.”

Outputs written to `docs/competition/digital-appendix/pincell_run/`:
`pincell_keff_vs_burnup.png`, `pincell_isotopics.png`, `depletion_results.h5`.